# Week 4 — CatBoost: Ordered Target Statistics & Ordered Boosting

> *Why naive target encoding leaks, why vanilla GBM has prediction shift, and how a single random permutation fixes both.*

## Learning objectives

By the end of this notebook, you will be able to:

1. Quantify the leakage induced by naive mean-target encoding on rare categories.
2. State and implement the ordered target statistic (TS) formula.
3. Explain prediction shift in vanilla boosting and how ordered boosting eliminates it.
4. Describe symmetric (oblivious) trees and the inference-time speed they enable.
5. Benchmark CatBoost's native categorical handling vs. manual encoding on a real high-cardinality problem.

## Outline

1. **Target leakage** in mean-target encoding
2. **Ordered target statistics** — the unbiased estimator
3. **Prediction shift** in vanilla gradient boosting
4. **Ordered boosting** — the gradient-side fix
5. **Symmetric (oblivious) trees** and their inference advantage
6. **End-to-end demo** on a high-cardinality categorical dataset
7. **Symmetric-tree inference speed benchmark**


## 1. The target-leakage problem

Categorical features must be encoded as numbers before a tree-based learner can split on them. The natural choice is **mean-target encoding** — replace each category $c$ with the mean target value on the rows where the feature equals $c$:

$$
\hat x_{i,j} = \frac{\sum_{k : x_{k,j} = x_{i,j}} y_k}{\sum_{k : x_{k,j} = x_{i,j}} 1}.
$$

This formula has a **fatal flaw**: row $i$'s encoding includes $y_i$ in the numerator. The encoder has been told $y_i$, and on rare categories (small denominator) the leaked $y_i$ dominates.

### Why the leak survives standard train/test splits

When you split into train and validation, you typically fit the encoder on the *training set only* and apply it to the validation set. But within the training set, every row's encoding still includes its own target. The model learns to rely on a feature that, at inference time, will not contain target information — producing a false sense of accuracy that collapses on truly held-out data.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))

from gradient_forge.catboost_internals.ordered_ts import (
    OrderedTargetStatistics, naive_mean_target_encode,
)
from gradient_forge.catboost_internals import CatBoostTrainer
from gradient_forge.data.loaders import make_categorical_dataset
from gradient_forge.utils import seed_everything, Stopwatch
seed_everything(42)


### 1.1 A dramatic leakage demonstration

Take a categorical feature uniformly random, a target uniformly random, and compute mean-target encoding. There is *no real signal* — yet the leaky encoding correlates almost perfectly with the target. The "signal" is entirely manufactured by the leak.


In [ ]:
rng = np.random.default_rng(0)
n = 1000
cats = rng.choice([f"cat_{i:02d}" for i in range(50)], size=n)  # 50 levels, ~20 rows each
y_random = rng.normal(size=n)                                   # PURE NOISE — no real signal

leaky = naive_mean_target_encode(cats, y_random)
ordered = OrderedTargetStatistics(smoothing=1.0, random_state=0).fit_transform(cats, y_random)

corr_leaky = abs(np.corrcoef(leaky, y_random)[0, 1])
corr_ord = abs(np.corrcoef(ordered, y_random)[0, 1])

print("Correlation between encoding and target (target is pure noise — should be ≈ 0):")
print(f"  naive mean-target encoding   : |corr| = {corr_leaky:.4f}   ← MASSIVE artificial signal")
print(f"  ordered target statistic     : |corr| = {corr_ord:.4f}    ← negligible, as it should be")


## 2. Ordered Target Statistics

Draw a random permutation $\sigma$ of the rows. For each row at position $p$ in $\sigma$, use **only** rows at positions $1, \dots, p - 1$ when computing the encoding:

$$
\hat x_{\sigma(p), j} = \frac{\displaystyle \sum_{k=1}^{p-1} \mathbb{1}\!\left[x_{\sigma(k),j} = x_{\sigma(p),j}\right] \cdot y_{\sigma(k)} \;+\; a \cdot p_0}{\displaystyle \sum_{k=1}^{p-1} \mathbb{1}\!\left[x_{\sigma(k),j} = x_{\sigma(p),j}\right] \;+\; a}.
$$

Here $p_0$ is a prior (usually $\bar y$) and $a > 0$ is a smoothing constant. By construction the encoding does not see $y_{\sigma(p)}$, so it is **unbiased under the permutation distribution**. Smoothing toward the prior shrinks encodings of rare levels toward the global mean, reducing variance.

### At inference

The full training set is available, so the **standard target statistic** is used:

$$
\hat x_{\text{test}, j}(c) = \frac{\sum_{k : x_{k,j} = c} y_k + a \cdot p_0}{\sum_{k : x_{k,j} = c} 1 + a}.
$$

This is leak-free at inference because the test row's target was never in the encoder's data.


### 2.1 Effect of permutation choice

A single permutation gives an unbiased but high-variance encoding. CatBoost actually maintains *multiple* permutations (typically 4) and combines them. Visualize the variance.


In [ ]:
n = 500
cats = rng.choice(["a", "b", "c", "d", "e"], size=n)
y = (cats == "a").astype(float) + rng.normal(scale=0.3, size=n)

# 50 encodings under different permutations.
encodings = np.array([
    OrderedTargetStatistics(smoothing=1.0, random_state=seed).fit_transform(cats, y)
    for seed in range(50)
])

# For a specific row, plot the distribution of encodings across permutations.
target_row = 0
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(encodings[:, target_row], bins=20, alpha=0.7, color="C0")
ax.axvline(y[target_row], color="red", linestyle="--",
           label=f"true y_i = {y[target_row]:.3f}")
ax.axvline(encodings[:, target_row].mean(), color="green", linestyle=":",
           label=f"mean across permutations = {encodings[:, target_row].mean():.3f}")
ax.set_xlabel("ordered TS encoding for row 0")
ax.set_title("Different permutations give different encodings — averaging reduces variance")
ax.legend(); plt.show()


## 3. Prediction shift in vanilla gradient boosting

The same kind of leakage that plagues mean-target encoding **also occurs in the gradient step of vanilla GBM**. The gradient

$$
g_i = \frac{\partial \ell(y_i, p)}{\partial p}\Bigg|_{p = F_{t-1}(x_i)}
$$

is computed with a model $F_{t-1}$ that was trained on data including $(x_i, y_i)$. The next tree $f_t$ therefore minimizes a target whose direction at $x_i$ has been informed by $y_i$ — a subtle form of leak.

Prokhorenkova et al. (2018) call this **prediction shift** and prove that the resulting estimator is *asymptotically biased*. The bias is small per iteration but compounds over many rounds, gradually pulling the model toward the training distribution at the expense of generalization.

### Ordered boosting

The fix mirrors ordered TS: under a permutation $\sigma$, train a sequence of models $\{M_1, \dots, M_n\}$ where $M_k$ is fit on the first $k$ rows. When computing $g_{\sigma(p)}$, use $M_{p-1}$ — a model that has never seen $y_{\sigma(p)}$.

A naive implementation would multiply training cost by $n$. CatBoost shares computation across multiple permutations and uses approximation schemes that bring the overhead down to a small multiplicative constant.


## 4. Symmetric (oblivious) trees

CatBoost's weak learners are not arbitrary regression trees — they are **oblivious** (a.k.a. symmetric): at every node at depth $d$, the split uses the **same** feature and threshold. A depth-$d$ oblivious tree is uniquely determined by:

- $d$ feature/threshold pairs
- a flat table of $2^d$ leaf values

### Why this matters at inference

Predicting on a row $x$ becomes:

1. Compare $x$ against each of the $d$ thresholds to produce a $d$-bit code.
2. Index into the leaf-value table.

That is **$d$ comparisons + one array lookup** per tree — branchless, SIMD-friendly, GPU-portable. On long-running production inference paths (e.g. ad-bidding at million-QPS), this is the difference between feasible and infeasible.

### Inductive bias

Oblivious trees are a **strict subset** of general trees, so they fit slightly less complex hypotheses. This is a *regularizer* — and it pairs naturally with the higher-variance ordered gradient estimator. Empirically, the symmetric structure costs little on tabular data while giving large inference-time benefits.


## 5. End-to-end demonstration on high-cardinality data

Build a synthetic e-commerce / banking-style dataset with two high-cardinality categorical features whose signal is recoverable only through good encoding.


In [ ]:
df, y = make_categorical_dataset(n_samples=10_000, n_categories=50, random_state=42)
print(df.head())
print(f"\nClass balance        : {pd.Series(y).value_counts(normalize=True).to_dict()}")
print(f"cat_a unique values  : {df['cat_a'].nunique()}")
print(f"cat_b unique values  : {df['cat_b'].nunique()}")

df_tr, df_va, y_tr, y_va = train_test_split(df, y, test_size=0.25, stratify=y, random_state=42)


### 5.1 Strategy A — manual mean-target encoding (the trap)

This is what an inexperienced practitioner does. The encoder uses training-set statistics, which sounds safe — but the encoding *within* the training set still leaks $y$ to the same row's features.


In [ ]:
def mean_target_encode_train_val(df_tr, df_va, y_tr, cols):
    enc_tr = df_tr.copy(); enc_va = df_va.copy()
    for c in cols:
        means = pd.Series(y_tr).groupby(df_tr[c].values).mean()
        enc_tr[c] = df_tr[c].map(means).astype(float)
        enc_va[c] = df_va[c].map(means).fillna(float(np.mean(y_tr))).astype(float)
    return enc_tr.to_numpy(), enc_va.to_numpy()

X_tr_lk, X_va_lk = mean_target_encode_train_val(df_tr, df_va, y_tr, ["cat_a", "cat_b"])

lr = LogisticRegression(max_iter=500).fit(X_tr_lk, y_tr)
auc_lk_tr = roc_auc_score(y_tr, lr.predict_proba(X_tr_lk)[:, 1])
auc_lk_va = roc_auc_score(y_va, lr.predict_proba(X_va_lk)[:, 1])
print(f"Manual mean-target encoding + LogReg:")
print(f"  train AUC = {auc_lk_tr:.4f}")
print(f"  val   AUC = {auc_lk_va:.4f}")
print(f"  train–val gap = {auc_lk_tr - auc_lk_va:+.4f}")


### 5.2 Strategy B — CatBoost native categorical handling

CatBoost handles ordered target statistics and ordered boosting internally. Just tell it which columns are categorical via `cat_features`.


In [ ]:
cat_idx = [df.columns.get_loc("cat_a"), df.columns.get_loc("cat_b")]
cat_model = CatBoostTrainer(
    task="binary", iterations=500, learning_rate=0.05, depth=6,
    cat_features=cat_idx, random_state=42,
).fit(df_tr.to_numpy(), y_tr,
      eval_set=(df_va.to_numpy(), y_va), early_stopping_rounds=30)

auc_cat_tr = roc_auc_score(y_tr, cat_model.predict_proba(df_tr.to_numpy())[:, 1])
auc_cat_va = roc_auc_score(y_va, cat_model.predict_proba(df_va.to_numpy())[:, 1])
print(f"CatBoost native categorical handling:")
print(f"  train AUC = {auc_cat_tr:.4f}")
print(f"  val   AUC = {auc_cat_va:.4f}")
print(f"  train–val gap = {auc_cat_tr - auc_cat_va:+.4f}")


In [ ]:
# Side-by-side summary.
pd.DataFrame([
    {"strategy": "manual mean-target + LogReg", "train AUC": auc_lk_tr, "val AUC": auc_lk_va,
     "gap": auc_lk_tr - auc_lk_va},
    {"strategy": "CatBoost native",              "train AUC": auc_cat_tr, "val AUC": auc_cat_va,
     "gap": auc_cat_tr - auc_cat_va},
])


**Reading the table.** CatBoost's smaller train–val gap is the visible signature of unbiased encoding. The manual encoder may even win on raw validation AUC for *this particular* dataset, but the gap is the diagnostic: it indicates that *some* of the training accuracy is illusory.


## 6. Symmetric-tree inference speed benchmark

Measure the per-row prediction latency of CatBoost (symmetric trees) vs. XGBoost / LightGBM (general trees) on the same task. Below we run both libraries with similar tree counts and depths and compare prediction time.


In [ ]:
from gradient_forge.xgboost_internals import XGBoostTrainer
from gradient_forge.lightgbm_internals import LightGBMTrainer

# Use only numeric features so all three libraries are on equal footing.
X_num = df[["num_a", "num_b"]].to_numpy()
y_num = y
X_tr_n, X_te_n, y_tr_n, y_te_n = train_test_split(X_num, y_num, test_size=0.3, stratify=y_num, random_state=42)

# Train all three with comparable settings.
common_iter = 500
xgb_m = XGBoostTrainer(task="binary", n_estimators=common_iter, max_depth=6,
                      learning_rate=0.05, random_state=42).fit(X_tr_n, y_tr_n)
lgb_m = LightGBMTrainer(task="binary", n_estimators=common_iter, num_leaves=64,
                       learning_rate=0.05, random_state=42).fit(X_tr_n, y_tr_n)
cat_m = CatBoostTrainer(task="binary", iterations=common_iter, depth=6,
                       learning_rate=0.05, random_state=42).fit(X_tr_n, y_tr_n)

# Time 5× repeated prediction on the test set.
X_pred = np.tile(X_te_n, (5, 1))   # ~15k rows
timings = {}
for name, m in [("XGBoost", xgb_m), ("LightGBM", lgb_m), ("CatBoost (oblivious)", cat_m)]:
    with Stopwatch() as sw:
        for _ in range(20):  # 20 repetitions → stable estimate
            _ = m.predict_proba(X_pred)[:, 1]
    timings[name] = sw.seconds

df_speed = pd.DataFrame({
    "library": list(timings.keys()),
    "total_inference_s": list(timings.values()),
    "us_per_row": [v / (X_pred.shape[0] * 20) * 1e6 for v in timings.values()],
})
df_speed


In [ ]:
# Bar chart.
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(df_speed["library"], df_speed["us_per_row"],
       color=["C0", "C1", "C3"])
ax.set_ylabel("μs per row (lower is better)")
ax.set_title("Inference latency — symmetric trees enable branchless prediction")
ax.grid(alpha=0.3, axis="y")
for i, v in enumerate(df_speed["us_per_row"]):
    ax.text(i, v + 0.01, f"{v:.2f} μs", ha="center", fontsize=10)
plt.show()


## 7. Exercises

1. **Cardinality sensitivity.** Vary `n_categories` from 5 to 500 and plot the train–val gap for both manual encoding and CatBoost. At what cardinality does the leakage become severe?
2. **Smoothing sweep.** With `n_categories=200`, vary `OrderedTargetStatistics(smoothing=...)` over `[0.1, 1.0, 10.0, 100.0]`. How does smoothing trade off bias and variance on rare categories?
3. **GPU acceleration.** Set `task_type="GPU"` and rerun the inference benchmark. How does the gap between symmetric and general trees change?
4. **Implement ordered TS.** Without consulting the source, re-implement `OrderedTargetStatistics.fit_transform` from the formula in Section 2. Verify it matches the shipped implementation to within numerical noise.

## Takeaways

- Naive target encoding **leaks** $y_i$ into row $i$'s features; the leakage is invisible to standard train–val splits.
- **Ordered target statistics** fix the leak by encoding under a random row permutation.
- **Vanilla GBM** has the analogous "prediction shift" bias; ordered boosting applies the same trick to gradient estimation.
- **Symmetric (oblivious) trees** are CatBoost's weak learners; they enable branchless prediction at the cost of a tiny inductive bias.
- The diagnostic for leakage is the **train–val AUC gap**, not the absolute validation score.

> **Next week:** automate hyperparameter tuning across all three libraries with Optuna's TPE sampler and GPU acceleration.
